# Visualize One Processed Scenario

Change `CONFIG["dataset"]` to `'waymo'` or `'av2'`, then run all cells. The notebook reads the processed ScenarioNet data directly and plots the HD map plus agent histories and futures for one scenario.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/mplconfig_tailrisk_notebooks")

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
from scenarionet.common_utils import read_dataset_summary, read_scenario

# Some processed AV2 pickles reference NumPy 2.x internal module paths.
# Apply the compatibility alias after importing MetaDrive/ScenarioNet.
sys.modules.setdefault("numpy._core", np.core)
sys.modules.setdefault("numpy._core.multiarray", np.core.multiarray)
sys.modules.setdefault("numpy._core.numeric", np.core.numeric)

print(f"Repo root: {REPO_ROOT}")

In [ ]:
CONFIG = {
    "dataset": "av2",          # "waymo" or "av2"
    "use_smoke_subset": False,   # True -> use repo-local tiny subsets from day 1
    "split": 'train',               # None -> validation for Waymo, val for AV2
    "scenario_index": 0,
    "scenario_id": None,         # Optional explicit scenario filename / id key from dataset_summary.pkl
    "dataset_root_override": None,
    "view_radius": 90.0,
    "center_on_sdc": True,
    "max_tracks": None,
    "show_track_ids": False,
    "line_width": 1.8,
}

CONFIG

In [ ]:
CATALOG = {
    "waymo": {
        "full_root": Path("/fs/nexus-projects/pc_driving/datasets/sn_womd"),
        "default_split": "validation",
        "smoke_root": REPO_ROOT / "data" / "smoke_subsets" / "waymo_smoke",
        "smoke_default_split": "validation",
    },
    "av2": {
        "full_root": Path("/fs/nexus-projects/pc_driving/datasets/argoverse2_sn"),
        "default_split": "val",
        "smoke_root": REPO_ROOT / "data" / "smoke_subsets" / "av2_smoke",
        "smoke_default_split": "val",
    },
}


def resolve_dataset_dir(config):
    key = config["dataset"].lower()
    if key not in CATALOG:
        raise ValueError(f"Unsupported dataset: {config['dataset']}")

    if config.get("dataset_root_override"):
        return Path(config["dataset_root_override"]).expanduser().resolve()

    entry = CATALOG[key]
    if config.get("use_smoke_subset"):
        root = entry["smoke_root"]
        split = config.get("split") or entry["smoke_default_split"]
    else:
        root = entry["full_root"]
        split = config.get("split") or entry["default_split"]
    return root / split


def choose_scenario_name(config, scenario_names):
    scenario_id = config.get("scenario_id")
    if scenario_id is not None:
        if scenario_id in scenario_names:
            return scenario_id
        raise KeyError(f"Scenario id {scenario_id!r} not found in selected split")

    scenario_index = int(config.get("scenario_index", 0))
    if scenario_index < 0 or scenario_index >= len(scenario_names):
        raise IndexError(f"scenario_index={scenario_index} is out of range for {len(scenario_names)} scenarios")
    return scenario_names[scenario_index]


def load_scenario_bundle(config):
    dataset_dir = resolve_dataset_dir(config)
    summary, scenario_names, mapping = read_dataset_summary(str(dataset_dir))
    scenario_name = choose_scenario_name(config, scenario_names)
    scenario = read_scenario(str(dataset_dir), mapping, scenario_name)
    return {
        "dataset_dir": dataset_dir,
        "scenario_name": scenario_name,
        "summary_entry": summary[scenario_name],
        "scenario": scenario,
    }


In [ ]:
def normalize_track_id(track_id):
    return str(track_id)


def target_track_ids(metadata):
    targets = set()
    for track_id, info in metadata.get("tracks_to_predict", {}).items():
        targets.add(normalize_track_id(info.get("track_id", track_id)))
    for track_id in metadata.get("objects_of_interest", []):
        targets.add(normalize_track_id(track_id))
    return targets


def valid_segments(positions, valid_mask):
    positions = np.asarray(positions)[..., :2]
    valid_mask = np.asarray(valid_mask).astype(bool).reshape(-1)
    segments = []
    start = None
    for idx, is_valid in enumerate(valid_mask):
        if is_valid and start is None:
            start = idx
        elif not is_valid and start is not None:
            if idx - start >= 2:
                segments.append(positions[start:idx])
            start = None
    if start is not None and len(valid_mask) - start >= 2:
        segments.append(positions[start:])
    return segments


def scenario_center(scenario, metadata, center_on_sdc=True):
    current_idx = int(metadata.get("current_time_index", 0))
    tracks = scenario["tracks"]

    if center_on_sdc:
        sdc_id = metadata.get("sdc_id")
        if sdc_id is not None and sdc_id in tracks:
            state = tracks[sdc_id]["state"]
            valid = np.asarray(state["valid"]).reshape(-1).astype(bool)
            if current_idx < len(valid) and valid[current_idx]:
                return np.asarray(state["position"])[current_idx, :2]

    current_positions = []
    for track in tracks.values():
        state = track["state"]
        valid = np.asarray(state["valid"]).reshape(-1).astype(bool)
        if current_idx < len(valid) and valid[current_idx]:
            current_positions.append(np.asarray(state["position"])[current_idx, :2])

    if current_positions:
        return np.mean(np.stack(current_positions, axis=0), axis=0)

    all_valid_positions = []
    for track in tracks.values():
        state = track["state"]
        valid = np.asarray(state["valid"]).reshape(-1).astype(bool)
        if valid.any():
            all_valid_positions.append(np.asarray(state["position"])[valid, :2])
    return np.mean(np.concatenate(all_valid_positions, axis=0), axis=0)


def map_style(map_type):
    map_type = str(map_type).upper()
    if "CROSSWALK" in map_type:
        return {"color": "#d8b365", "linewidth": 1.2, "linestyle": "-"}
    if "STOP_SIGN" in map_type or "SPEED_BUMP" in map_type:
        return {"color": "#b15928", "linewidth": 1.2, "linestyle": "-"}
    if "ROAD_EDGE" in map_type or "BOUNDARY" in map_type or "MEDIAN" in map_type:
        return {"color": "#7f7f7f", "linewidth": 1.0, "linestyle": "-"}
    if "LANE" in map_type or "ROAD_LINE" in map_type:
        return {"color": "#c7c7c7", "linewidth": 0.9, "linestyle": "-"}
    return {"color": "#dddddd", "linewidth": 0.8, "linestyle": "-"}


def track_color(track_type, role):
    if role == "sdc":
        return "#ff7f0e"
    if role == "target":
        return "#d62728"
    if role == "interest":
        return "#9467bd"

    track_type = str(track_type).upper()
    if "PEDESTRIAN" in track_type:
        return "#2ca02c"
    if "CYCLIST" in track_type or "MOTOR" in track_type:
        return "#17becf"
    if "VEHICLE" in track_type:
        return "#1f77b4"
    return "#7f7f7f"


def draw_map(ax, map_features):
    for feature in map_features.values():
        polyline = np.asarray(feature.get("polyline", []))
        if polyline.ndim != 2 or len(polyline) < 2:
            continue
        style = map_style(feature.get("type", "UNKNOWN"))
        ax.plot(
            polyline[:, 0],
            polyline[:, 1],
            color=style["color"],
            linewidth=style["linewidth"],
            linestyle=style["linestyle"],
            alpha=0.9,
            zorder=1,
        )


def draw_tracks(ax, scenario, config):
    metadata = scenario["metadata"]
    tracks = list(scenario["tracks"].items())
    current_idx = int(metadata.get("current_time_index", 0))
    sdc_id = metadata.get("sdc_id")
    targets = target_track_ids(metadata)
    interests = {normalize_track_id(x) for x in metadata.get("objects_of_interest", [])}

    if config.get("max_tracks") is not None:
        tracks = tracks[: int(config["max_tracks"])]

    for track_id, track in tracks:
        state = track["state"]
        positions = np.asarray(state["position"])
        valid = np.asarray(state["valid"]).reshape(-1).astype(bool)
        norm_id = normalize_track_id(track_id)

        if norm_id == normalize_track_id(sdc_id):
            role = "sdc"
        elif norm_id in targets:
            role = "target"
        elif norm_id in interests:
            role = "interest"
        else:
            role = "other"

        color = track_color(track.get("type", "UNKNOWN"), role)
        history_mask = valid.copy()
        history_mask[current_idx + 1:] = False
        future_mask = valid.copy()
        future_mask[:current_idx] = False

        line_width = 2.6 if role in {"sdc", "target"} else config.get("line_width", 1.8)
        alpha = 1.0 if role in {"sdc", "target"} else 0.7

        for segment in valid_segments(positions, history_mask):
            ax.plot(segment[:, 0], segment[:, 1], color=color, linewidth=line_width, alpha=alpha, zorder=3)

        for segment in valid_segments(positions, future_mask):
            ax.plot(
                segment[:, 0],
                segment[:, 1],
                color=color,
                linewidth=max(line_width - 0.4, 1.0),
                linestyle="--",
                alpha=alpha,
                zorder=4,
            )

        if current_idx < len(valid) and valid[current_idx]:
            current_xy = positions[current_idx, :2]
            marker_size = 36 if role in {"sdc", "target"} else 20
            ax.scatter(current_xy[0], current_xy[1], color=color, s=marker_size, zorder=5)
            if config.get("show_track_ids"):
                ax.text(current_xy[0], current_xy[1], norm_id, fontsize=7, color=color, zorder=6)


def plot_scenario(scenario, config):
    metadata = scenario["metadata"]
    center_xy = scenario_center(scenario, metadata, center_on_sdc=config.get("center_on_sdc", True))
    view_radius = float(config.get("view_radius", 90.0))

    fig, ax = plt.subplots(figsize=(10, 10))
    draw_map(ax, scenario["map_features"])
    draw_tracks(ax, scenario, config)

    ax.set_aspect("equal")
    ax.set_xlim(center_xy[0] - view_radius, center_xy[0] + view_radius)
    ax.set_ylim(center_xy[1] - view_radius, center_xy[1] + view_radius)
    ax.axis("off")

    legend_handles = [
        Line2D([0], [0], color="#ff7f0e", lw=2.6, label="SDC"),
        Line2D([0], [0], color="#d62728", lw=2.6, label="Tracks to predict"),
        Line2D([0], [0], color="#1f77b4", lw=1.8, label="Other vehicles"),
        Line2D([0], [0], color="#2ca02c", lw=1.8, label="Pedestrians"),
        Line2D([0], [0], color="#17becf", lw=1.8, label="Cyclists / motorcyclists"),
        Line2D([0], [0], color="#7f7f7f", lw=1.0, label="Map boundaries"),
        Line2D([0], [0], color="black", lw=1.2, linestyle="--", label="Future segment"),
    ]
    ax.legend(handles=legend_handles, loc="upper right", frameon=False)

    title = f"{config['dataset'].upper()} | {metadata.get('scenario_id', scenario.get('id', 'unknown'))}"
    ax.set_title(title)
    return fig, ax


In [ ]:
CONFIG["scenario_index"] +=1

In [ ]:
006c3bbe-6d0b-4317-9b06-0b29ea498d19

In [ ]:
bundle = load_scenario_bundle(CONFIG)
scenario = bundle["scenario"]
metadata = scenario["metadata"]

print(f"Dataset dir: {bundle['dataset_dir']}")
print(f"Scenario key: {bundle['scenario_name']}")
print(f"Scenario id: {metadata.get('scenario_id', scenario.get('id'))}")
print(f"Tracks: {len(scenario['tracks'])}")
print(f"Map features: {len(scenario['map_features'])}")
print(f"Current time index: {metadata.get('current_time_index')}")
print(f"SDC id: {metadata.get('sdc_id')}")
print(f"Tracks to predict: {sorted(target_track_ids(metadata))}")

In [ ]:
scenario.keys()

In [ ]:
fig, ax = plot_scenario(scenario, CONFIG)
plt.show()

In [ ]:
fig, ax = plot_scenario(scenario, CONFIG)
plt.show()

In [ ]:
fig, ax = plot_scenario(scenario, CONFIG)
plt.show()